# 04 — Outcome Prediction (Hired vs Not Hired)

Implements the benchmark methodology from **Teinemaa et al. (2019) — Outcome-Oriented
Predictive Process Monitoring: Review and Benchmark** (doi:10.1145/3301300).

**Core idea**: evaluate predictive accuracy not just at full-trace level but across every prefix
length k, using multiple encoding schemes. For a prefix of length k, only the first k events
of each case are visible, simulating real-time decision support.

| Component | Choice | pm4py function |
|-----------|--------|----------------|
| Case-level split | Random 75/25 | `pm.split_train_test` |
| Prefix generation | Prefix-length bucketing | `pm.get_prefixes_from_log` |
| Boolean encoding | Binary activity occurrence | `pm.extract_outcome_enriched_dataframe` |
| Frequency encoding | Activity occurrence counts | `pm.extract_features_dataframe(count_occurrences=True)` |
| Classifiers | RF, XGBoost, LightGBM | scikit-learn / xgboost / lightgbm |
| Primary metric | AUC-ROC per prefix length | sklearn |

## 1. Imports

In [20]:
import pandas as pd
import numpy as np
import matplotlib
import pm4py as pm
import random
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score,
    precision_recall_curve
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ── SMOTE-NC toggle ───────────────────────────────────────────────────────────
# Set True to oversample minority class in training only (never val/test).
# SMOTE-NC handles mixed feature types: binary activity flags (categorical)
# and continuous timing features (@@sojourn_time, etc.).
USE_SMOTENC = True

# ── imbalanced-learn import (fail fast with a clear message) ─────────────────
if USE_SMOTENC:
    try:
        from imblearn.over_sampling import SMOTENC, SMOTE
    except ImportError as _e:
        raise ImportError(
            f"Original error: {_e}"
        ) from _e

## 2. Load & Clean

In [21]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv('data/' + 
    'event_log_consolidated.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp']
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Events     : {len(df):,}')
print(f'Cases      : {df["Case_id"].nunique():,}')
print(f'Activities : {df["Step"].nunique()}')

Events     : 1,133,314
Cases      : 279,458
Activities : 58


## 3. Format for pm4py

In [22]:
df = pm.format_dataframe(
    df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp'
)
print('pm4py aliases added: case:concept:name | concept:name | time:timestamp')

pm4py aliases added: case:concept:name | concept:name | time:timestamp


In [23]:
df

,Job_Requisition_Code,Candidate_id,Recruiting Agency,Region,Country,Job Family,Job Family Group,Case_id,CW,Evergreen,...,Completed By,Step,Disposition Reason,All Stages for Candidate Current and Completed,timestamp,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,10fb9be7d6ea,000086ee80c8,NaN,4,15,57,16,000086ee80c8 - 10fb9be7d6ea,0,0,...,6746,Review Decision,Candidate Withdrew,Declined by Candidate\n\nInterview\n\nStart,2024-10-17 15:21:25.626000+00:00,000086ee80c8 - 10fb9be7d6ea,Review Decision,2024-10-17 15:21:25.626000+00:00,0,0
1,10fb9be7d6ea,000086ee80c8,NaN,4,15,57,16,000086ee80c8 - 10fb9be7d6ea,0,0,...,7566,Start,Candidate Withdrew,Declined by Candidate\n\nInterview\n\nStart,2024-10-17 15:21:25.626000+00:00,000086ee80c8 - 10fb9be7d6ea,Start,2024-10-17 15:21:25.626000+00:00,1,0
2,10fb9be7d6ea,000086ee80c8,NaN,4,15,57,16,000086ee80c8 - 10fb9be7d6ea,0,0,...,6746,Schedule Interview,Candidate Withdrew,Declined by Candidate\n\nInterview\n\nStart,2024-10-17 15:22:10.434000+00:00,000086ee80c8 - 10fb9be7d6ea,Schedule Interview,2024-10-17 15:22:10.434000+00:00,2,0
3,10fb9be7d6ea,000086ee80c8,NaN,4,15,57,16,000086ee80c8 - 10fb9be7d6ea,0,0,...,6746,Make Interview Decision,Candidate Withdrew,Declined by Candidate\n\nInterview\n\nStart,2024-10-17 15:22:10.434000+00:00,000086ee80c8 - 10fb9be7d6ea,Make Interview Decision,2024-10-17 15:22:10.434000+00:00,3,0
4,10fb9be7d6ea,000086ee80c8,NaN,4,15,57,16,000086ee80c8 - 10fb9be7d6ea,0,0,...,7566,Recruiter Interview,Candidate Withdrew,Declined by Candidate\n\nInterview\n\nStart,2024-10-17 15:22:10.434000+00:00,000086ee80c8 - 10fb9be7d6ea,Recruiter Interview,2024-10-17 15:22:10.434000+00:00,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1123416,497d4a37969a,ffff721f23cd,NaN,4,15,59,0,ffff721f23cd - 497d4a37969a,0,0,...,4959,Review Decision,Candidate never contacted,Rejected\n\nStart,2024-05-07 09:20:54.288000+00:00,ffff721f23cd - 497d4a37969a,Review Decision,2024-05-07 09:20:54.288000+00:00,1123416,279455
1123417,b88ab174591d,ffffbb818934,NaN,4,15,31,4,ffffbb818934 - b88ab174591d,0,0,...,7222,Review Decision,Candidate never contacted,Rejected\n\nStart,2024-07-15 10:29:49.345000+00:00,ffffbb818934 - b88ab174591d,Review Decision,2024-07-15 10:29:49.345000+00:00,1123417,279456
1123418,b88ab174591d,ffffbb818934,NaN,4,15,31,4,ffffbb818934 - b88ab174591d,0,0,...,7566,Start,Candidate never contacted,Rejected\n\nStart,2024-07-15 10:29:49.345000+00:00,ffffbb818934 - b88ab174591d,Start,2024-07-15 10:29:49.345000+00:00,1123418,279456
1123419,f3e8560e5358,fffff0debbec,NaN,1,10,139,13,fffff0debbec - f3e8560e5358,0,0,...,7566,Start,Candidate never contacted,Rejected\n\nStart,2025-02-19 16:00:55.948000+00:00,fffff0debbec - f3e8560e5358,Start,2025-02-19 16:00:55.948000+00:00,1123419,279457


## 4. Train / Validation / Test Split — Chronological 60 / 20 / 20

Case-level **chronological** split following Weytjens et al. (2021), who show that random
splits inflate AUC by leaking future process behaviour into training data.

* Cases are ordered by their **first event timestamp**.
* First 60 % → **train**, next 20 % → **validation** (early stopping), last 20 % → **test**.
* Cutoff dates are printed below for transparency.

In [24]:
# ── Strict chronological 60/20/20 split (Weytjens et al. 2021 + leakage fix) ─
# Train : cases fully completed BEFORE train cutoff (no post-cutoff events leak in)
# Val   : cases starting AFTER train cutoff, completed before val cutoff
#         + straddlers (started pre-train-cutoff, ended in val window): post-cutoff events only
# Test  : cases starting AFTER val cutoff
#         + straddlers (started pre-val-cutoff, ended after): post-cutoff events only
case_times = df.groupby('case:concept:name').agg(
    case_start=('time:timestamp', 'min'),
    case_end  =('time:timestamp', 'max'),
).sort_values('case_start')

n            = len(case_times)
train_cutoff = case_times.iloc[int(n * 0.60)]['case_start']
val_cutoff   = case_times.iloc[int(n * 0.80)]['case_start']

train_ids = case_times[case_times['case_end']   <  train_cutoff].index
val_ids   = case_times[(case_times['case_start'] >= train_cutoff) &
                        (case_times['case_end']   <  val_cutoff)].index
test_ids  = case_times[case_times['case_start'] >= val_cutoff].index

train_df = df[df['case:concept:name'].isin(train_ids)].copy()

straddle_tv = case_times[(case_times['case_start'] <  train_cutoff) &
                          (case_times['case_end']   >= train_cutoff) &
                          (case_times['case_end']   <  val_cutoff)].index
val_df = pd.concat([
    df[df['case:concept:name'].isin(val_ids)],
    df[df['case:concept:name'].isin(straddle_tv) & (df['time:timestamp'] >= train_cutoff)],
]).sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

straddle_vt = case_times[(case_times['case_start'] <  val_cutoff) &
                          (case_times['case_end']   >= val_cutoff)].index
test_df = pd.concat([
    df[df['case:concept:name'].isin(test_ids)],
    df[df['case:concept:name'].isin(straddle_vt) & (df['time:timestamp'] >= val_cutoff)],
]).sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

print(f'Train cutoff: {train_cutoff.date()} | Val cutoff: {val_cutoff.date()}')
print(f'Train — {train_df["case:concept:name"].nunique():,} cases  |  {len(train_df):,} events')
print(f'Val   — {val_df["case:concept:name"].nunique():,} cases  |  {len(val_df):,} events')
print(f'Test  — {test_df["case:concept:name"].nunique():,} cases  |  {len(test_df):,} events')


Train cutoff: 2024-09-12 | Val cutoff: 2025-08-29
Train — 162,133 cases  |  691,878 events
Val   — 60,212 cases  |  195,485 events
Test  — 57,113 cases  |  206,146 events


In [25]:

# ── Recruiting Agency encoding ────────────────────────────────────────────────
# Text column; NaN means no external agency was involved.
# Fit LabelEncoder on train only; unseen agencies in val/test map to 0 ("No Agency").

from sklearn.preprocessing import LabelEncoder

AGENCY_COL = 'Recruiting Agency'

def encode_agency(df_split, encoder=None, fit=False):
    """Label-encode Recruiting Agency. NaN → 'No Agency' (always index 0)."""
    s = df_split.groupby('case:concept:name')[AGENCY_COL].first().fillna('No Agency')
    if fit:
        encoder = LabelEncoder()
        encoder.fit(s)
    # Map unseen labels to 'No Agency' (index 0)
    known = set(encoder.classes_)
    s_mapped = s.apply(lambda x: x if x in known else 'No Agency')
    return encoder.transform(s_mapped), encoder

# Fit on train
train_agency_arr, agency_encoder = encode_agency(train_df, fit=True)
val_agency_arr,   _              = encode_agency(val_df,   encoder=agency_encoder)
test_agency_arr,  _              = encode_agency(test_df,  encoder=agency_encoder)

# Build per-split Series indexed by case:concept:name
def _agency_series(df_split, arr):
    case_ids = df_split.groupby('case:concept:name')[AGENCY_COL].first().index
    return pd.Series(arr, index=case_ids, name='agency_enc')

train_agency = _agency_series(train_df, train_agency_arr)
val_agency   = _agency_series(val_df,   val_agency_arr)
test_agency  = _agency_series(test_df,  test_agency_arr)

n_agencies = len(agency_encoder.classes_)
print(f'Recruiting Agency: {n_agencies} unique values (incl. "No Agency")')
print(f'No-agency cases — train: {(train_agency == 0).sum():,} | '
      f'val: {(val_agency == 0).sum():,} | test: {(test_agency == 0).sum():,}')


Recruiting Agency: 61 unique values (incl. "No Agency")
No-agency cases — train: 1 | val: 0 | test: 0


## 5. Extract Labels

`hired` is case-level (same for all events in a case). Labels must be extracted **before**
leaky activity removal so no cases are dropped from the label index.

In [26]:
y_train_series = (
    train_df.groupby('case:concept:name')['hired']
    .first().fillna(0).astype(int)
)
y_val_series = (
    val_df.groupby('case:concept:name')['hired']
    .first().fillna(0).astype(int)
)
y_test_series = (
    test_df.groupby('case:concept:name')['hired']
    .first().fillna(0).astype(int)
)
print(f'Train hire rate : {y_train_series.mean():.2%}  ({y_train_series.sum():,} / {len(y_train_series):,})')
print(f'Val   hire rate : {y_val_series.mean():.2%}  ({y_val_series.sum():,} / {len(y_val_series):,})')
print(f'Test  hire rate : {y_test_series.mean():.2%}  ({y_test_series.sum():,} / {len(y_test_series):,})')

Train hire rate : 2.42%  (3,921 / 162,133)
Val   hire rate : 1.54%  (925 / 60,212)
Test  hire rate : 1.35%  (771 / 57,113)


## 6. Remove Leaky Activities

Activities whose hire rate on the **training set** exceeds 0.85 are post-decision steps
(background checks, offer acceptance) that encode the outcome. Removed from both splits
before any feature extraction.

In [27]:
LEAKY_THRESHOLD = 0.85

act_hire_rate = (
    train_df.groupby('concept:name')['hired']
    .apply(lambda x: x.astype(float).mean())
    .sort_values(ascending=False)
)
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)

print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()
val_df  = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()

print(f'\nActivities remaining : {train_df["concept:name"].nunique()}')
print(f'Train events : {len(train_df):,}  |  Test events : {len(test_df):,}')

Leaky activities removed (16):
  1.000  FurstPerson Integration
  1.000  Ready for Hire
  0.997  Additional Offer Packet Document(s)
  0.990  Review Company and Office Guides
  0.985  Candidate Experience Survey
  0.983  Recruiting Process Survey
  0.974  Automatically Unpost Jobs
  0.967  Confirm No Manager Survey
  0.927  LATAM Standard Questionnaire (2024)
  0.924  Colombia Standard Questionnaire (2024)
  0.900  Additional Background / Reference Check
  0.891  Renegotiate Offer
  0.885  Make Background Check Decision
  0.885  Background Check (LATAM)
  0.884  Set Background Check Status
  0.851  Consolidated Approval by Initiator

Activities remaining : 35
Train events : 667,960  |  Test events : 201,869


### 7. Encoding Functions (Teinemaa et al. 2019 + Mannhardt 2024)

Three encoding schemes are compared:

| Encoding | Description | Source |
|----------|-------------|--------|
| **Boolean** | Binary flag per activity: 1 if occurred in prefix, 0 otherwise | `pm.extract_outcome_enriched_dataframe` |
| **Frequency** | Count per activity: occurrences in prefix | `pm.extract_features_dataframe(count_occurrences=True)` |
| **Succession** | Binary flag per activity bigram (a→b): did this consecutive pair occur? | shift-based computation on DataFrame; equivalent to pm4py's `str_evsucc_attr` |

All encodings are augmented with six static case attributes (Region, Country, Job Family, Job Family Group, CW, Evergreen). Boolean encoding additionally includes timing features from `extract_outcome_enriched_dataframe`.

Prefix generation uses `pm.get_prefixes_from_log` (prefix-length bucketing).

In [29]:
PM_COLS        = ['case:concept:name', 'concept:name', 'time:timestamp']
SKIP_COLS      = {'case:concept:name', 'concept:name', 'time:timestamp', '@@diff_start_end'}
CASE_ATTR_COLS = ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen', 'agency_enc']

# Pre-compute static case attributes (case-level, safe regardless of prefix length)
case_attrs_train = (
    train_df.groupby('case:concept:name')[['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']]
    .first().fillna(0).astype(float)
    .join(train_agency.rename('agency_enc').astype(float))
)
case_attrs_val = (
    val_df.groupby('case:concept:name')[['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']]
    .first().fillna(0).astype(float)
    .join(val_agency.rename('agency_enc').astype(float))
)
case_attrs_test = (
    test_df.groupby('case:concept:name')[['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']]
    .first().fillna(0).astype(float)
    .join(test_agency.rename('agency_enc').astype(float))
)


def get_prefix_pm(df_clean, k):
    """Truncate each case to its first k events via pm4py; 'all' returns full traces."""
    if k == 'all':
        return df_clean[PM_COLS].copy()
    prefix = pm.get_prefixes_from_log(
        df_clean, length=k, case_id_key='case:concept:name'
    )
    return prefix[PM_COLS].copy()


def bool_encoding(prefix_pm):
    """
    Boolean encoding (Teinemaa et al. §3.2).
    pm.extract_outcome_enriched_dataframe → one binary column per activity
    (did it occur at least once?) + case-level timing features (@@cols).
    Returns a case-indexed DataFrame (one row per case).
    """
    enriched = pm.extract_outcome_enriched_dataframe(
        prefix_pm,
        activity_key='concept:name',
        timestamp_key='time:timestamp',
        case_id_key='case:concept:name',
    )
    feat_cols = [c for c in enriched.columns if c not in SKIP_COLS]
    return (
        enriched
        .drop_duplicates('case:concept:name')
        .set_index('case:concept:name')[feat_cols]
        .sort_index()
    )


def freq_encoding(prefix_pm):
    """
    Frequency encoding (Teinemaa et al. §3.2).
    pm.extract_features_dataframe(count_occurrences=True) → one integer-count
    column per activity (how many times it occurred in the prefix).
    Returns a case-indexed DataFrame (one row per case).
    """
    fea_df = pm.extract_features_dataframe(
        prefix_pm,
        activity_key='concept:name',
        timestamp_key='time:timestamp',
        case_id_key='case:concept:name',
        include_case_id=True,
        count_occurrences=True,
    )
    return fea_df.set_index('case:concept:name').sort_index()


def succ_encoding(prefix_pm):
    """
    Succession (bigram) encoding — from Mannhardt (2024) lecture4 notebook.
    For each consecutive activity pair (a → b) in the prefix, records whether
    it occurred (binary flag). Equivalent to pm4py's str_evsucc_attr parameter
    in log_to_features.apply but computed directly on the DataFrame for efficiency.

    Captures sequential patterns that boolean/frequency encodings miss:
    e.g. 'Start >> Review Decision' vs 'Review Decision >> Start' are distinct features.
    Returns a case-indexed DataFrame (one row per case).

    Edge cases:
    - k=1: no bigrams possible → returns zero-column DataFrame with all case IDs.
    - Short traces at larger k: cases with only 1 event get zeros for all bigram columns.
    Both are handled by reindexing to the full set of case IDs before returning.
    """
    all_case_ids = sorted(prefix_pm['case:concept:name'].unique())

    df = prefix_pm.copy()
    df['prev_act'] = df.groupby('case:concept:name')['concept:name'].shift(1)
    df = df.dropna(subset=['prev_act'])   # drop first event per case (no predecessor)

    if df.empty:
        # k=1 (or all cases are single-event): no bigrams at all.
        # Return an empty-column DataFrame so build_Xy still has all case rows.
        return pd.DataFrame(
            index=pd.Index(all_case_ids, name='case:concept:name')
        )

    df['bigram'] = df['prev_act'] + ' >> ' + df['concept:name']
    bigram_counts = (
        df.groupby(['case:concept:name', 'bigram'])
        .size()
        .unstack(fill_value=0)
    )
    # Ensure every case appears — single-event cases within the prefix get zeros
    bigram_counts = bigram_counts.reindex(all_case_ids, fill_value=0)
    bigram_counts.index.name = 'case:concept:name'
    return bigram_counts


def build_Xy(X_pm, case_attrs_split, y_series, train_cols=None):
    """
    Align pm4py feature columns to training vocabulary (fill 0 for unseen),
    join static case attributes, and retrieve labels aligned to the case index.
    train_cols must NOT include CASE_ATTR_COLS — those are always joined here.
    """
    if train_cols is not None:
        X_pm = X_pm.reindex(columns=train_cols, fill_value=0)
    X = X_pm.join(case_attrs_split, how='left').fillna(0)
    y = y_series.reindex(X.index).fillna(0).values.astype(np.int32)
    return X.values.astype(np.float32), y, list(X.columns)


print('Encoding helpers defined: bool_encoding | freq_encoding | succ_encoding')
print(f'Case attributes : {CASE_ATTR_COLS}')


Encoding helpers defined: bool_encoding | freq_encoding | succ_encoding
Case attributes : ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen', 'agency_enc']


In [30]:
def apply_smotenc(X, y, feat_names, random_state=42):
    """
    Apply SMOTE-NC to training data only when USE_SMOTENC is True.

    Categorical columns: binary activity-presence flags (concept:name_*)
    and integer-encoded case attributes (Region, Country, Job Family, etc.).
    Continuous columns: pm4py timing features (@@sojourn_time, @@waiting_time, etc.)
    and bigram succession flags treated as continuous (they are 0/1 but SMOTE handles them).

    Returns resampled (X_res, y_res) or unchanged (X, y) if USE_SMOTENC is False.
    SMOTENC and SMOTE are imported at notebook startup (code-imports cell).
    """
    if not USE_SMOTENC:
        return X, y

    CASE_ATTRS = {'Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen'}
    cat_mask = np.array([
        name.startswith('concept:name_') or name in CASE_ATTRS
        for name in feat_names
    ])
    cat_indices = np.where(cat_mask)[0].tolist()

    if len(cat_indices) == 0 or len(cat_indices) == len(feat_names):
        # All continuous or all categorical — fall back to standard SMOTE
        sm = SMOTE(random_state=random_state)
    else:
        sm = SMOTENC(categorical_features=cat_indices, random_state=random_state)

    X_res, y_res = sm.fit_resample(X, y)
    minority_orig = int(y.sum())
    minority_res  = int(y_res.sum())
    print(f'  SMOTE-NC: {len(y):,} → {len(y_res):,} samples '
          f'(minority: {minority_orig:,} → {minority_res:,})')
    return X_res.astype(np.float32), y_res.astype(np.int32)

## 8. Prefix-Length Sweep (Teinemaa et al. core evaluation)

For each prefix length k and each encoding, a **separate LightGBM** classifier is trained
and evaluated (prefix-length bucketing). Replicates the paper's AUC-vs-prefix-length
analysis across three encoding schemes.

Prefix lengths: **[1, 3, 5, 10, 20, 'all']** — Encodings: **Boolean | Frequency | Succession**

In [31]:
import lightgbm as lgb

PREFIX_LENGTHS = [1, 3, 5, 10, 20, 'all']
ENCODINGS      = [('Boolean',    bool_encoding),
                  ('Frequency',  freq_encoding),
                  ('Succession', succ_encoding)]   # added from Mannhardt (2024)
sweep_results  = {}   # (encoding_name, k) -> {'AUC': float, 'PR-AUC': float}

# When SMOTE-NC has already balanced the training set, class_weight='balanced'
# would double-correct for imbalance — use it only when SMOTE-NC is off.
_class_weight = None if USE_SMOTENC else 'balanced'

for enc_name, enc_fn in ENCODINGS:
    print(f'\n── {enc_name} encoding ────────────────────')
    for k in PREFIX_LENGTHS:
        # ── prefix generation (pm4py) ─────────────────────────────────────────
        train_prefix_pm = get_prefix_pm(train_df, k)
        val_prefix_pm   = get_prefix_pm(val_df,   k)
        test_prefix_pm  = get_prefix_pm(test_df,  k)

        # ── feature extraction ────────────────────────────────────────────────
        X_tr_pm    = enc_fn(train_prefix_pm)
        train_cols = list(X_tr_pm.columns)   # pm4py feature cols only (no case attrs)
        X_va_pm    = enc_fn(val_prefix_pm)
        X_te_pm    = enc_fn(test_prefix_pm)

        # ── build matrices (case_attrs joined inside build_Xy) ────────────────
        X_tr, y_tr, feat_names = build_Xy(X_tr_pm, case_attrs_train, y_train_series)
        X_va, y_va, _          = build_Xy(X_va_pm, case_attrs_val,   y_val_series,
                                          train_cols=train_cols)
        X_te, y_te, _          = build_Xy(X_te_pm, case_attrs_test,  y_test_series,
                                          train_cols=train_cols)

        # ── oversample training set only (val/test untouched) ─────────────────
        X_tr, y_tr = apply_smotenc(X_tr, y_tr, feat_names, random_state=RANDOM_STATE)

        # ── LightGBM per (encoding × prefix length) with early stopping ───────
        clf = LGBMClassifier(
            n_estimators=300, class_weight=_class_weight,
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
        )
        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(20, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        proba = clf.predict_proba(X_te)[:, 1]

        auc   = roc_auc_score(y_te, proba)
        prauc = average_precision_score(y_te, proba)
        sweep_results[(enc_name, k)] = {'AUC': auc, 'PR-AUC': prauc}
        print(f'  k={str(k):5s}  AUC={auc:.4f}  PR-AUC={prauc:.4f}')


── Boolean encoding ────────────────────
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=1      AUC=0.8602  PR-AUC=0.1538
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=3      AUC=0.9531  PR-AUC=0.4573
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=5      AUC=0.9742  PR-AUC=0.4908
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=10     AUC=0.9819  PR-AUC=0.5239
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=20     AUC=0.9938  PR-AUC=0.8256
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=all    AUC=0.9971  PR-AUC=0.8440

── Frequency encoding ────────────────────
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=1      AUC=0.7237  PR-AUC=0.0680
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=3      AUC=0.9138  PR-AUC=0.4321
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
  k=5      AUC=0.9453  P

## 9. AUC vs Prefix Length Plot

The key figure from Teinemaa et al.: predictive accuracy improves as more process
context becomes available. Comparing boolean vs frequency encoding.

In [32]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
enc_styles = {
    'Boolean'   : ('tab:blue',   'o-'),
    'Frequency' : ('tab:orange', 's--'),
    'Succession': ('tab:green',  '^:'),   # added
}

for metric, ax, ylabel in [
    ('AUC',    axes[0], 'ROC-AUC'),
    ('PR-AUC', axes[1], 'PR-AUC (Avg. Precision)'),
]:
    for enc_name, (color, style) in enc_styles.items():
        x_vals = list(range(len(PREFIX_LENGTHS)))
        y_vals = [sweep_results[(enc_name, k)][metric] for k in PREFIX_LENGTHS]
        ax.plot(x_vals, y_vals, style, color=color, label=enc_name, linewidth=2, markersize=7)

    ax.set_xticks(range(len(PREFIX_LENGTHS)))
    ax.set_xticklabels([str(k) for k in PREFIX_LENGTHS])
    ax.set_xlabel('Prefix length k')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs Prefix Length')
    ax.legend()
    ax.set_ylim(0.5, 1.0)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Outcome Prediction — Teinemaa et al. (2019) Prefix-Length Evaluation\n'
    'Succession encoding: Mannhardt (2024)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('figs/outcome_auc_vs_prefix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/outcome_auc_vs_prefix.png')

Saved figs/outcome_auc_vs_prefix.png


## 10. Sweep Summary Table

In [33]:
rows = []
for (enc, k), metrics in sweep_results.items():
    rows.append({'Encoding': enc, 'Prefix k': str(k), **metrics})
sweep_df = pd.DataFrame(rows).pivot(index='Prefix k', columns='Encoding')
sweep_df.index = pd.CategoricalIndex(
    sweep_df.index,
    categories=[str(k) for k in PREFIX_LENGTHS],
    ordered=True
)
sweep_df = sweep_df.sort_index()
print('=== AUC / PR-AUC by Prefix Length and Encoding ===')
print(sweep_df.to_string(float_format='{:.4f}'.format))

=== AUC / PR-AUC by Prefix Length and Encoding ===
             AUC                       PR-AUC                     
Encoding Boolean Frequency Succession Boolean Frequency Succession
Prefix k                                                          
1         0.8602    0.7237     0.7313  0.1538    0.0680     0.0809
3         0.9531    0.9138     0.8929  0.4573    0.4321     0.4107
5         0.9742    0.9453     0.9276  0.4908    0.5146     0.4825
10        0.9819    0.9578     0.9642  0.5239    0.5429     0.5961
20        0.9938    0.9818     0.9734  0.8256    0.7782     0.7955
all       0.9971    0.9832     0.9791  0.8440    0.8205     0.8428


## 11. Full-Trace Models (k = 'all')

At full prefix (all events visible), compare **RF, XGBoost, LightGBM** under all three
encoding schemes. Succession encoding's columns are activity bigrams (`A >> B`), so its
feature space is larger (~n² potential pairs) but sparser.

In [34]:
full_train_pm = train_df[PM_COLS].copy()
full_val_pm   = val_df[PM_COLS].copy()
full_test_pm  = test_df[PM_COLS].copy()

print('Extracting full-trace boolean features...')
X_bool_tr_pm = bool_encoding(full_train_pm)
bool_train_cols = list(X_bool_tr_pm.columns)
X_bool_va_pm = bool_encoding(full_val_pm)
X_bool_te_pm = bool_encoding(full_test_pm)

print('Extracting full-trace frequency features...')
X_freq_tr_pm = freq_encoding(full_train_pm)
freq_train_cols = list(X_freq_tr_pm.columns)
X_freq_va_pm = freq_encoding(full_val_pm)
X_freq_te_pm = freq_encoding(full_test_pm)

print('Extracting full-trace succession features...')
X_succ_tr_pm = succ_encoding(full_train_pm)
succ_train_cols = list(X_succ_tr_pm.columns)
X_succ_va_pm = succ_encoding(full_val_pm)
X_succ_te_pm = succ_encoding(full_test_pm)

# Assemble — case_attrs joined by build_Xy; train_cols must NOT include them
X_bool_tr, y_tr, bool_feat_names = build_Xy(X_bool_tr_pm, case_attrs_train, y_train_series)
X_bool_va, y_va, _               = build_Xy(X_bool_va_pm, case_attrs_val,   y_val_series,
                                             train_cols=bool_train_cols)
X_bool_te, y_te, _               = build_Xy(X_bool_te_pm, case_attrs_test,  y_test_series,
                                             train_cols=bool_train_cols)

X_freq_tr, _,    freq_feat_names = build_Xy(X_freq_tr_pm, case_attrs_train, y_train_series)
X_freq_va, _,    _               = build_Xy(X_freq_va_pm, case_attrs_val,   y_val_series,
                                             train_cols=freq_train_cols)
X_freq_te, _,    _               = build_Xy(X_freq_te_pm, case_attrs_test,  y_test_series,
                                             train_cols=freq_train_cols)

X_succ_tr, _,    succ_feat_names = build_Xy(X_succ_tr_pm, case_attrs_train, y_train_series)
X_succ_va, _,    _               = build_Xy(X_succ_va_pm, case_attrs_val,   y_val_series,
                                             train_cols=succ_train_cols)
X_succ_te, _,    _               = build_Xy(X_succ_te_pm, case_attrs_test,  y_test_series,
                                             train_cols=succ_train_cols)

# ── SMOTE-NC: oversample each training split (val/test remain untouched) ──────
print('\nApplying SMOTE-NC to boolean training data...')
X_bool_tr, y_tr_bool = apply_smotenc(X_bool_tr, y_tr, bool_feat_names, random_state=RANDOM_STATE)

print('Applying SMOTE-NC to frequency training data...')
X_freq_tr, y_tr_freq = apply_smotenc(X_freq_tr, y_tr, freq_feat_names, random_state=RANDOM_STATE)

print('Applying SMOTE-NC to succession training data...')
X_succ_tr, y_tr_succ = apply_smotenc(X_succ_tr, y_tr, succ_feat_names, random_state=RANDOM_STATE)

pos_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
print(f'\nBoolean    : train {X_bool_tr.shape}  |  val {X_bool_va.shape}  |  test {X_bool_te.shape}')
print(f'Frequency  : train {X_freq_tr.shape}  |  val {X_freq_va.shape}  |  test {X_freq_te.shape}')
print(f'Succession : train {X_succ_tr.shape}  |  val {X_succ_va.shape}  |  test {X_succ_te.shape}')
print(f'pos_weight (XGB, original y_tr): {pos_weight:.1f}')

Extracting full-trace boolean features...
Extracting full-trace frequency features...
Extracting full-trace succession features...

Applying SMOTE-NC to boolean training data...
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
Applying SMOTE-NC to frequency training data...
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)
Applying SMOTE-NC to succession training data...
  SMOTE-NC: 162,132 → 316,424 samples (minority: 3,920 → 158,212)

Boolean    : train (316424, 47)  |  val (60108, 47)  |  test (57035, 47)
Frequency  : train (316424, 42)  |  val (60108, 42)  |  test (57035, 42)
Succession : train (316424, 550)  |  val (60108, 550)  |  test (57035, 550)
pos_weight (XGB, original y_tr): 40.4


In [35]:
def run_models(X_train, X_val, X_test, y_train, y_val, y_test, encoding_label):
    """Train RF / XGB / LGBM and return results, probabilities, and fitted models.

    XGB and LGBM use the validation set for early stopping (patience=20 rounds).
    Class-imbalance correction is applied only when SMOTE-NC is OFF:
      - RF / LGBM: class_weight='balanced'
      - XGB:       scale_pos_weight = n_neg / n_pos  (original train distribution)
    When SMOTE-NC is ON the training set is already balanced, so these weights
    would double-correct and are set to neutral (None / 1.0).
    """
    if USE_SMOTENC:
        rf_class_weight  = None
        xgb_spw          = 1.0
        lgbm_class_weight = None
    else:
        rf_class_weight   = 'balanced'
        xgb_spw           = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        lgbm_class_weight = 'balanced'

    models = {
        'RF': RandomForestClassifier(
            n_estimators=500, class_weight=rf_class_weight,
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        'XGBoost': XGBClassifier(
            n_estimators=500, scale_pos_weight=xgb_spw,
            n_jobs=-1, random_state=RANDOM_STATE,
            verbosity=0, eval_metric='logloss',
            early_stopping_rounds=20,
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=500, class_weight=lgbm_class_weight,
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
        ),
    }
    results, probas, trained = {}, {}, {}
    for name, clf in models.items():
        print(f'  [{encoding_label}] {name}...')
        if name == 'XGBoost':
            clf.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False,
            )
        elif name == 'LightGBM':
            clf.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb.early_stopping(20, verbose=False),
                    lgb.log_evaluation(period=-1),
                ],
            )
        else:
            clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        pred  = (proba >= 0.5).astype(int)
        results[name] = {
            'Accuracy' : accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred, zero_division=0),
            'Recall'   : recall_score(y_test, pred, zero_division=0),
            'F1'       : f1_score(y_test, pred, zero_division=0),
            'ROC-AUC'  : roc_auc_score(y_test, proba),
            'PR-AUC'   : average_precision_score(y_test, proba),
        }
        probas[name]  = proba
        trained[name] = clf
        m = results[name]
        print(f'    F1={m["F1"]:.4f}  ROC-AUC={m["ROC-AUC"]:.4f}  PR-AUC={m["PR-AUC"]:.4f}')
    return results, probas, trained


print('=== Boolean Encoding (k=all) ===')
bool_results, bool_probas, bool_models = run_models(
    X_bool_tr, X_bool_va, X_bool_te, y_tr_bool, y_va, y_te, 'Boolean'
)

print('\n=== Frequency Encoding (k=all) ===')
freq_results, freq_probas, freq_models = run_models(
    X_freq_tr, X_freq_va, X_freq_te, y_tr_freq, y_va, y_te, 'Frequency'
)

print('\n=== Succession Encoding (k=all) ===')
succ_results, succ_probas, succ_models = run_models(
    X_succ_tr, X_succ_va, X_succ_te, y_tr_succ, y_va, y_te, 'Succession'
)

=== Boolean Encoding (k=all) ===
  [Boolean] RF...
    F1=0.8494  ROC-AUC=0.9899  PR-AUC=0.8402
  [Boolean] XGBoost...
    F1=0.8098  ROC-AUC=0.9967  PR-AUC=0.8288
  [Boolean] LightGBM...
    F1=0.8264  ROC-AUC=0.9971  PR-AUC=0.8440

=== Frequency Encoding (k=all) ===
  [Frequency] RF...
    F1=0.7895  ROC-AUC=0.9870  PR-AUC=0.7929
  [Frequency] XGBoost...
    F1=0.7860  ROC-AUC=0.9852  PR-AUC=0.8194
  [Frequency] LightGBM...
    F1=0.7106  ROC-AUC=0.9832  PR-AUC=0.8205

=== Succession Encoding (k=all) ===
  [Succession] RF...
    F1=0.8463  ROC-AUC=0.9767  PR-AUC=0.8503
  [Succession] XGBoost...
    F1=0.8016  ROC-AUC=0.9793  PR-AUC=0.8343
  [Succession] LightGBM...
    F1=0.8060  ROC-AUC=0.9791  PR-AUC=0.8428


## 12. Results Table (k = 'all')

In [36]:
all_enc_results = {
    'Boolean'   : bool_results,
    'Frequency' : freq_results,
    'Succession': succ_results,
}
rows = []
for enc, res in all_enc_results.items():
    for model, metrics in res.items():
        rows.append({'Encoding': enc, 'Model': model, **metrics})

full_res = pd.DataFrame(rows).set_index(['Encoding', 'Model'])
print('\n=== OUTCOME PREDICTION RESULTS (k=all) ===')
print(full_res.to_string(float_format='{:.4f}'.format))


=== OUTCOME PREDICTION RESULTS (k=all) ===
                     Accuracy  Precision  Recall     F1  ROC-AUC  PR-AUC
Encoding   Model                                                        
Boolean    RF          0.9962     0.8146  0.8874 0.8494   0.9899  0.8402
           XGBoost     0.9952     0.7830  0.8384 0.8098   0.9967  0.8288
           LightGBM    0.9955     0.7822  0.8759 0.8264   0.9971  0.8440
Frequency  RF          0.9942     0.7074  0.8932 0.7895   0.9870  0.7929
           XGBoost     0.9941     0.7035  0.8903 0.7860   0.9852  0.8194
           LightGBM    0.9918     0.6247  0.8240 0.7106   0.9832  0.8205
Succession RF          0.9961     0.8136  0.8817 0.8463   0.9767  0.8503
           XGBoost     0.9948     0.7529  0.8571 0.8016   0.9793  0.8343
           LightGBM    0.9950     0.7652  0.8514 0.8060   0.9791  0.8428


## 13. Precision-Recall Curves (k = 'all')

PR curves for both encodings × all classifiers. PR-AUC is the primary metric for
imbalanced datasets (~2% hire rate).

In [37]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
baseline  = y_te.mean()

enc_panels = [
    ('Boolean',    bool_probas,  bool_results),
    ('Frequency',  freq_probas,  freq_results),
    ('Succession', succ_probas,  succ_results),
]
for ax, (enc_label, probas_dict, results_dict) in zip(axes, enc_panels):
    for name, proba in probas_dict.items():
        prec, rec, _ = precision_recall_curve(y_te, proba)
        ap = results_dict[name]['PR-AUC']
        ax.plot(rec, prec, label=f'{name} (PR-AUC={ap:.3f})')
    ax.axhline(baseline, color='grey', linestyle='--', linewidth=1,
               label=f'Baseline ({baseline:.2%})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{enc_label} encoding (k=all)')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Precision-Recall Curves — Outcome Prediction (k=all)', fontsize=13)
plt.tight_layout()
plt.savefig('figs/outcome_pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/outcome_pr_curves.png')

Saved figs/outcome_pr_curves.png


### 14. Feature Importance — LightGBM
#### Best Encoding (k = 'all')\n\nThe best-performing encoding (highest LightGBM PR-AUC) is selected automatically.\nSuccession features are named `A >> B`; boolean/frequency are named `concept:name_A`."

In [38]:
# Select encoding with highest LightGBM PR-AUC
enc_prauc = {
    'Boolean'   : bool_results['LightGBM']['PR-AUC'],
    'Frequency' : freq_results['LightGBM']['PR-AUC'],
    'Succession': succ_results['LightGBM']['PR-AUC'],
}
best_enc = max(enc_prauc, key=enc_prauc.get)
print(f'Best encoding by LightGBM PR-AUC: {best_enc}')
for enc, val in enc_prauc.items():
    print(f'  {enc:12s}: PR-AUC={val:.4f}{"  ✓" if enc == best_enc else ""}')

feat_map   = {'Boolean': bool_feat_names, 'Frequency': freq_feat_names, 'Succession': succ_feat_names}
model_map  = {'Boolean': bool_models,     'Frequency': freq_models,     'Succession': succ_models}
feat_names = feat_map[best_enc]
lgbm_best  = model_map[best_enc]['LightGBM']

importances = lgbm_best.feature_importances_
top_k   = 20
top_idx  = np.argsort(importances)[::-1][:top_k]
top_imp  = importances[top_idx]
top_feat = [feat_names[i] for i in top_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_k), top_imp[::-1], color='steelblue')
ax.set_yticks(range(top_k))
ax.set_yticklabels(top_feat[::-1], fontsize=9)
ax.set_xlabel('Feature importance (split gain)')
ax.set_title(f'Top-20 Features — LightGBM, {best_enc} Encoding, k=all')
plt.tight_layout()
plt.savefig('figs/outcome_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/outcome_feature_importance.png')

Best encoding by LightGBM PR-AUC: Boolean
  Boolean     : PR-AUC=0.8440  ✓
  Frequency   : PR-AUC=0.8205
  Succession  : PR-AUC=0.8428
Saved figs/outcome_feature_importance.png
